#### Sprint 3 – Hyperparameter Tuning & Model Optimization

##### Tasks

##### Tune the model with GridSearchCV

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"D:\MACHINE LEARNING\archive (4)\Reviews.csv")

In [3]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [4]:
df.shape

(568454, 10)

In [5]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [6]:
import re
df = df.dropna(subset=["Text", "Score"]).copy()
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["Cleaned_Review"]=df["Text"].apply(clean_text)
df["Cleaned_Review"]

0         i have bought several of the vitality canned d...
1         product arrived labeled as jumbo salted peanut...
2         this is a confection that has been around a fe...
3         if you are looking for the secret ingredient i...
4         great taffy at a great price there was a wide ...
                                ...                        
568449    great for sesame chicken this is a good if not...
568450    i m disappointed with the flavor the chocolate...
568451    these stars are small so you can give of those...
568452    these are the best treats for training and rew...
568453    i am very satisfied product is as advertised i...
Name: Cleaned_Review, Length: 568454, dtype: object

In [7]:
from sklearn.model_selection import train_test_split

X = df["Cleaned_Review"]
y = df["Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 454763
Testing rows: 113691


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)


Train shape: (454763, 10000)
Test shape: (113691, 10000)


In [9]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)
print(y_train.shape)
print(y_test.shape)

(454763, 10000)
(113691, 10000)
(454763,)
(113691,)


In [11]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

param_grid = {
    "C": [0.1, 1, 10],
    "class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_tfidf, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", grid_search.best_score_)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'C': 10, 'class_weight': None}
Best CV F1-score: 0.7254952723892139


##### Compare baseline and tuned models

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Create and train the baseline model
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

# Get predictions
baseline_pred = baseline_model.predict(X_test_tfidf)

# Tuned model predictions
tuned_model = grid_search.best_estimator_
tuned_pred = tuned_model.predict(X_test_tfidf)

# Compare weighted F1 scores
print(
    "Baseline F1:",
    f1_score(y_test, baseline_pred, average="weighted")
)

print(
    "Tuned F1:",
    f1_score(y_test, tuned_pred, average="weighted")
)

Baseline F1: 0.7195967878719075
Tuned F1: 0.7287584233295981


##### Analyze feature importance

In [14]:
feature_names = tfidf.get_feature_names_out()
class_index = list(tuned_model.classes_).index(5)

coefficients = tuned_model.coef_[class_index]

top_indices = np.argsort(coefficients)[-20:][::-1]

print(feature_names[top_indices])

['hooked' 'great' 'delicious' 'skeptical' 'perfect' 'highly' 'best'
 'pleasantly' 'excellent' 'amazing' 'wonderful' 'love' 'companion'
 'awesome' 'exceeded' 'fantastic' 'exquisite' 'worried' 'amazed' 'yum']


##### Save the final model and vectorizer

In [15]:
import joblib

joblib.dump(tuned_model, "sentiment_model.pkl")
joblib.dump(tfidf, "vectorizer.pkl")

['vectorizer.pkl']

●	Provide a summary explaining why the selected model performs the best

The selected model performs the best because it achieves the best overall performance among the evaluated machine learning models. It provides a good balance between accuracy, precision, recall, and F1-score. The confusion matrix and classification report help us understand how effectively the model predicts sentiment classes and minimizes misclassifications. Therefore, based on the evaluation results, the selected model is considered the most suitable for predicting sentiment from Amazon customer reviews.

#### Sprint 4 – Streamlit Application

##### Install Streamlit

In [1]:
%pip install streamlit joblib

Note: you may need to restart the kernel to use updated packages.


##### Create app.py

In [ ]:
import streamlit as st
import joblib
import re
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

# Load saved model and vectorizer
model = joblib.load("sentiment_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

# Prepare text preprocessing
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    words = text.split()
    words = [
        word for word in words
        if word not in stop_words or word == "not"
    ]
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)

st.title("Amazon Review Sentiment Analysis")
st.write("Enter a product review to predict its rating.")

review = st.text_area("Enter your review")

if st.button("Predict"):
    if review.strip():
        cleaned_review = preprocess_text(review)
        features = vectorizer.transform([cleaned_review])
        prediction = int(model.predict(features)[0])

        if prediction in [4, 5]:
            sentiment = "Positive"
        elif prediction in [1, 2]:
            sentiment = "Negative"
        else:
            sentiment = "Neutral"

        st.success(f"Predicted Rating: {prediction}/5")
        st.write(f"Predicted Sentiment: {sentiment}")
    else:
        st.warning("Please enter a review first.")


##### Run the application

In [ ]:
streamlit run app.py

##### Test the application

Test 1 — Positive review

Expected category

“Excellent quality, very tasty, and worth the money.”
Expected sentiment category: Positive. The exact predicted rating depends on your model.


Test 2 — Negative review

Expected category

“Very poor quality. I did not like this product.”
Expected sentiment category: Negative. The exact predicted rating depends on your model.


Test 3 — Neutral review

Expected category

“The package arrived today. The product is as described.”
Expected category is uncertain because neutral sentiment is not directly labelled in the original five-star rating system.

●	Display predicted rating

Predicted rating    

1 or 2	                

3	                   

4 or 5	                

●	Display predicted sentiment category

Sentiment Category

Negative

Neutral

Positive